## Import và tạo SparkSession

In [1]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    sum as spark_sum,
    avg,
    round,
    desc,
    current_timestamp,
    row_number, when
)
from pyspark.sql.window import Window

# Ép Spark chạy dưới quyền root để HDFS cho phép ghi dữ liệu
os.environ["HADOOP_USER_NAME"] = "root"

spark = SparkSession.builder \
    .appName("Retail_Silver_To_Gold_Layer") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,mysql:mysql-connector-java:8.0.33") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("SparkSession đã khởi tạo thành công.")

SparkSession đã khởi tạo thành công.


## Khai báo đường dẫn HDFS và MySQL

In [2]:
# HDFS PATHS
SILVER_PATH = "hdfs://namenode:9000/data/silver/retail/transactions_cleaned"

GOLD_BASE_PATH = "hdfs://namenode:9000/data/gold/retail"

GOLD_PRODUCT_RECOMMENDATIONS_PATH = f"{GOLD_BASE_PATH}/product_recommendations"
GOLD_SALES_BY_DATE_PATH = f"{GOLD_BASE_PATH}/sales_by_date"
GOLD_SALES_BY_MONTH_PATH = f"{GOLD_BASE_PATH}/sales_by_month"
GOLD_SALES_BY_HOUR_PATH = f"{GOLD_BASE_PATH}/sales_by_hour"
GOLD_TOP_PRODUCTS_PATH = f"{GOLD_BASE_PATH}/top_products"
GOLD_TOP_CUSTOMERS_PATH = f"{GOLD_BASE_PATH}/top_customers"
GOLD_SALES_BY_COUNTRY_PATH = f"{GOLD_BASE_PATH}/sales_by_country"
GOLD_HOLIDAY_IMPACT_PATH = f"{GOLD_BASE_PATH}/holiday_impact"
GOLD_WEATHER_IMPACT_PATH = f"{GOLD_BASE_PATH}/weather_impact"

print("Silver path:", SILVER_PATH)
print("Gold base path:", GOLD_BASE_PATH)

# ============================================================

# MYSQL CONFIG
db_url = "jdbc:mysql://mysql:3306/retail_analytics"

db_props = {
    "driver": "com.mysql.cj.jdbc.Driver",
    "user": "retail_user",
    "password": "retail_pass"
}

Silver path: hdfs://namenode:9000/data/silver/retail/transactions_cleaned
Gold base path: hdfs://namenode:9000/data/gold/retail


## Đọc dữ liệu từ Silver Layer

In [3]:
try:
    silver_df = spark.read.parquet(SILVER_PATH)
    silver_df.cache()

    silver_count = silver_df.count()

    print("Đã đọc dữ liệu từ Silver Layer thành công.")
    print(f"Số dòng Silver: {silver_count}")

except Exception as e:
    print(f"Lỗi khi đọc dữ liệu từ Silver Layer: {e}")
    raise e

silver_df.printSchema()
silver_df.show(10, truncate=False)

Đã đọc dữ liệu từ Silver Layer thành công.
Số dòng Silver: 524876
root
 |-- DateKey: integer (nullable = true)
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- batch_id: integer (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- TotalPrice: double (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Rainfall: double (nullable = true)
 |-- Snowfall: double (nullable = true)
 |-- HolidayName: string (nullable = true)
 |-- IsHoliday: boolean (nullable = true)

+--------+---------+---------+-----------------------------------+------

## Kiểm tra nhanh dữ liệu Silver

In [4]:
print("===== SILVER QUICK CHECK =====")

print("Tổng số dòng:", silver_df.count())
print("Số hóa đơn riêng biệt:", silver_df.select("InvoiceNo").distinct().count())
print("Số sản phẩm riêng biệt:", silver_df.select("StockCode").distinct().count())
print("Số khách hàng riêng biệt:", silver_df.select("CustomerID").distinct().count())

silver_df.select(
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "UnitPrice",
    "CustomerID",
    "Country",
    "DateKey",
    "Hour",
    "TotalPrice"
).show(10, truncate=False)

===== SILVER QUICK CHECK =====
Tổng số dòng: 524876
Số hóa đơn riêng biệt: 19960
Số sản phẩm riêng biệt: 3922
Số khách hàng riêng biệt: 4339
+---------+---------+-----------------------------------+--------+---------+----------+--------------+--------+----+----------+
|InvoiceNo|StockCode|Description                        |Quantity|UnitPrice|CustomerID|Country       |DateKey |Hour|TotalPrice|
+---------+---------+-----------------------------------+--------+---------+----------+--------------+--------+----+----------+
|561226   |PADS     |PADS TO MATCH ALL CUSHIONS         |1       |0.001    |15618     |United Kingdom|20110726|10  |0.001     |
|560359   |16008    |SMALL FOLDING SCISSOR(POINTED EDGE)|1       |0.12     |14159     |United Kingdom|20110718|11  |0.12      |
|550298   |20668    |DISCO BALL CHRISTMAS DECORATION    |1       |0.12     |17346     |United Kingdom|20110415|16  |0.12      |
|537222   |15034    |PAPER POCKET TRAVELING FAN         |1       |0.14     |14404     |Unit

## Gold Business Insights

### gold_sales_by_date

In [5]:
gold_sales_by_date = silver_df.groupBy(
    "DateKey",
    "Year",
    "Month",
    "Day",
    "DayOfWeek"
).agg(
    count("*").alias("NumberOfTransactions"),
    countDistinct("InvoiceNo").alias("NumberOfInvoices"),
    spark_sum("Quantity").alias("TotalQuantity"),
    round(spark_sum("TotalPrice"), 2).alias("TotalRevenue"),
    round(avg("TotalPrice"), 2).alias("AverageLineRevenue")
).withColumn(
    "UpdatedAt",
    current_timestamp()
).orderBy("DateKey")

print("===== GOLD: SALES BY DATE =====")
gold_sales_by_date.show(10, truncate=False)

===== GOLD: SALES BY DATE =====
+--------+----+-----+---+---------+--------------------+----------------+-------------+------------+------------------+--------------------------+
|DateKey |Year|Month|Day|DayOfWeek|NumberOfTransactions|NumberOfInvoices|TotalQuantity|TotalRevenue|AverageLineRevenue|UpdatedAt                 |
+--------+----+-----+---+---------+--------------------+----------------+-------------+------------+------------------+--------------------------+
|20101201|2010|12   |1  |4        |3028                |127             |26818        |58776.79    |19.41             |2026-05-07 05:57:32.012712|
|20101202|2010|12   |2  |5        |2022                |142             |31264        |47629.42    |23.56             |2026-05-07 05:57:32.012712|
|20101203|2010|12   |3  |6        |2138                |68              |16158        |46898.63    |21.94             |2026-05-07 05:57:32.012712|
|20101205|2010|12   |5  |1        |2603                |88              |16242        

### gold_sales_by_month

In [6]:
gold_sales_by_month = silver_df.groupBy(
    "Year",
    "Month"
).agg(
    count("*").alias("NumberOfTransactions"),
    countDistinct("InvoiceNo").alias("NumberOfInvoices"),
    spark_sum("Quantity").alias("TotalQuantity"),
    round(spark_sum("TotalPrice"), 2).alias("TotalRevenue"),
    round(avg("TotalPrice"), 2).alias("AverageLineRevenue")
).withColumn(
    "UpdatedAt",
    current_timestamp()
).orderBy("Year", "Month")

print("===== GOLD: SALES BY MONTH =====")
gold_sales_by_month.show(10, truncate=False)

===== GOLD: SALES BY MONTH =====
+----+-----+--------------------+----------------+-------------+------------+------------------+--------------------------+
|Year|Month|NumberOfTransactions|NumberOfInvoices|TotalQuantity|TotalRevenue|AverageLineRevenue|UpdatedAt                 |
+----+-----+--------------------+----------------+-------------+------------+------------------+--------------------------+
|2010|12   |40991               |1559            |358019       |821452.73   |20.04             |2026-05-07 05:57:33.745983|
|2011|1    |34060               |1086            |387099       |689811.61   |20.25             |2026-05-07 05:57:33.745983|
|2011|2    |26882               |1100            |282934       |522545.56   |19.44             |2026-05-07 05:57:33.745983|
|2011|3    |35497               |1454            |376599       |716215.26   |20.18             |2026-05-07 05:57:33.745983|
|2011|4    |28882               |1246            |307953       |536968.49   |18.59             |202

### gold_sales_by_hour

In [7]:
gold_sales_by_hour = silver_df.groupBy(
    "Hour"
).agg(
    count("*").alias("NumberOfTransactions"),
    countDistinct("InvoiceNo").alias("NumberOfInvoices"),
    spark_sum("Quantity").alias("TotalQuantity"),
    round(spark_sum("TotalPrice"), 2).alias("TotalRevenue"),
    round(avg("TotalPrice"), 2).alias("AverageLineRevenue")
).withColumn(
    "UpdatedAt",
    current_timestamp()
).orderBy("Hour")

print("===== GOLD: SALES BY HOUR =====")
gold_sales_by_hour.show(10, truncate=False)

===== GOLD: SALES BY HOUR =====
+----+--------------------+----------------+-------------+------------+------------------+--------------------------+
|Hour|NumberOfTransactions|NumberOfInvoices|TotalQuantity|TotalRevenue|AverageLineRevenue|UpdatedAt                 |
+----+--------------------+----------------+-------------+------------+------------------+--------------------------+
|6   |1                   |1               |1            |4.25        |4.25              |2026-05-07 05:57:34.738644|
|7   |379                 |29              |15370        |31059.21    |81.95             |2026-05-07 05:57:34.738644|
|8   |8797                |566             |158649       |283750.68   |32.26             |2026-05-07 05:57:34.738644|
|9   |33684               |1484            |517985       |990054.99   |29.39             |2026-05-07 05:57:34.738644|
|10  |47597               |2361            |812678       |1444814.77  |30.36             |2026-05-07 05:57:34.738644|
|11  |55419             

In [8]:
gold_sales_by_hour.orderBy(desc("NumberOfTransactions")).show(10, truncate=False)

+----+--------------------+----------------+-------------+------------+------------------+--------------------------+
|Hour|NumberOfTransactions|NumberOfInvoices|TotalQuantity|TotalRevenue|AverageLineRevenue|UpdatedAt                 |
+----+--------------------+----------------+-------------+------------+------------------+--------------------------+
|12  |75986               |3220            |836642       |1439324.66  |18.94             |2026-05-07 05:57:35.591368|
|15  |75665               |2336            |626989       |1350333.31  |17.85             |2026-05-07 05:57:35.591368|
|13  |69992               |2753            |703450       |1260658.2   |18.01             |2026-05-07 05:57:35.591368|
|14  |65057               |2457            |597056       |1177907.52  |18.11             |2026-05-07 05:57:35.591368|
|11  |55419               |2396            |671278       |1236558.44  |22.31             |2026-05-07 05:57:35.591368|
|16  |52992               |1335            |330609      

### gold_top_products

In [9]:
gold_top_products = silver_df.groupBy(
    "StockCode",
    "Description"
).agg(
    count("*").alias("NumberOfTransactions"),
    countDistinct("InvoiceNo").alias("NumberOfInvoices"),
    spark_sum("Quantity").alias("TotalQuantitySold"),
    round(spark_sum("TotalPrice"), 2).alias("TotalRevenue"),
    round(avg("UnitPrice"), 2).alias("AverageUnitPrice")
).withColumn(
    "UpdatedAt",
    current_timestamp()
).orderBy(desc("TotalRevenue"))

print("===== GOLD: TOP PRODUCTS =====")
gold_top_products.show(20, truncate=False)

===== GOLD: TOP PRODUCTS =====
+---------+----------------------------------+--------------------+----------------+-----------------+------------+----------------+--------------------------+
|StockCode|Description                       |NumberOfTransactions|NumberOfInvoices|TotalQuantitySold|TotalRevenue|AverageUnitPrice|UpdatedAt                 |
+---------+----------------------------------+--------------------+----------------+-----------------+------------+----------------+--------------------------+
|DOT      |DOTCOM POSTAGE                    |706                 |706             |706              |206248.77   |292.14          |2026-05-07 05:57:36.329262|
|22423    |REGENCY CAKESTAND 3 TIER          |2007                |1988            |13851            |174156.54   |13.98           |2026-05-07 05:57:36.329262|
|23843    |PAPER CRAFT , LITTLE BIRDIE       |1                   |1               |80995            |168469.6    |2.08            |2026-05-07 05:57:36.329262|
|85123A  

### gold_top_customers

In [10]:
gold_top_customers = silver_df.filter(
    col("CustomerID") != -1
).groupBy(
    "CustomerID",
    "Country"
).agg(
    count("*").alias("NumberOfTransactions"),
    countDistinct("InvoiceNo").alias("NumberOfInvoices"),
    spark_sum("Quantity").alias("TotalQuantityPurchased"),
    round(spark_sum("TotalPrice"), 2).alias("TotalRevenue"),
    round(avg("TotalPrice"), 2).alias("AverageLineRevenue")
).withColumn(
    "UpdatedAt",
    current_timestamp()
).orderBy(desc("TotalRevenue"))

print("===== GOLD: TOP CUSTOMERS =====")
gold_top_customers.show(10, truncate=False)

===== GOLD: TOP CUSTOMERS =====
+----------+--------------+--------------------+----------------+----------------------+------------+------------------+--------------------------+
|CustomerID|Country       |NumberOfTransactions|NumberOfInvoices|TotalQuantityPurchased|TotalRevenue|AverageLineRevenue|UpdatedAt                 |
+----------+--------------+--------------------+----------------+----------------------+------------+------------------+--------------------------+
|14646     |Netherlands   |2076                |73              |196915                |280206.02   |134.97            |2026-05-07 05:57:38.705536|
|18102     |United Kingdom|431                 |60              |64124                 |259657.3    |602.45            |2026-05-07 05:57:38.705536|
|17450     |United Kingdom|336                 |46              |69973                 |194390.79   |578.54            |2026-05-07 05:57:38.705536|
|16446     |United Kingdom|3                   |2               |80997          

### gold_sales_by_country

In [11]:
gold_sales_by_country = silver_df.groupBy(
    "Country"
).agg(
    count("*").alias("NumberOfTransactions"),
    countDistinct("InvoiceNo").alias("NumberOfInvoices"),
    countDistinct("CustomerID").alias("NumberOfCustomers"),
    spark_sum("Quantity").alias("TotalQuantity"),
    round(spark_sum("TotalPrice"), 2).alias("TotalRevenue"),
    round(avg("TotalPrice"), 2).alias("AverageLineRevenue")
).withColumn(
    "UpdatedAt",
    current_timestamp()
).orderBy(desc("TotalRevenue"))

print("===== GOLD: SALES BY COUNTRY =====")
gold_sales_by_country.show(10, truncate=False)

===== GOLD: SALES BY COUNTRY =====
+--------------+--------------------+----------------+-----------------+-------------+------------+------------------+--------------------------+
|Country       |NumberOfTransactions|NumberOfInvoices|NumberOfCustomers|TotalQuantity|TotalRevenue|AverageLineRevenue|UpdatedAt                 |
+--------------+--------------------+----------------+-----------------+-------------+------------+------------------+--------------------------+
|United Kingdom|479983              |18019           |3921             |4646603      |9001192.24  |18.75             |2026-05-07 05:57:40.151432|
|Netherlands   |2359                |94              |9                |200361       |285446.34   |121.0             |2026-05-07 05:57:40.151432|
|EIRE          |7879                |288             |4                |147007       |283140.52   |35.94             |2026-05-07 05:57:40.151432|
|Germany       |9025                |457             |94               |119154       |228

### gold_holiday_impact

In [12]:
gold_holiday_impact = silver_df.groupBy(
    "IsHoliday",
    "HolidayName"
).agg(
    count("*").alias("NumberOfTransactions"),
    countDistinct("InvoiceNo").alias("NumberOfInvoices"),
    spark_sum("Quantity").alias("TotalQuantity"),
    round(spark_sum("TotalPrice"), 2).alias("TotalRevenue"),
    round(avg("TotalPrice"), 2).alias("AverageLineRevenue")
).withColumn(
    "UpdatedAt",
    current_timestamp()
).orderBy(desc("TotalRevenue"))

print("===== GOLD: HOLIDAY IMPACT =====")
gold_holiday_impact.show(20, truncate=False)

===== GOLD: HOLIDAY IMPACT =====
+---------+-----------+--------------------+----------------+-------------+-------------+------------------+--------------------------+
|IsHoliday|HolidayName|NumberOfTransactions|NumberOfInvoices|TotalQuantity|TotalRevenue |AverageLineRevenue|UpdatedAt                 |
+---------+-----------+--------------------+----------------+-------------+-------------+------------------+--------------------------+
|false    |Regular Day|524876              |19960           |5572117      |1.064155895E7|20.27             |2026-05-07 05:57:41.828666|
+---------+-----------+--------------------+----------------+-------------+-------------+------------------+--------------------------+



### gold_weather_impact

In [13]:
silver_weather_df = silver_df.withColumn(
    "WeatherCondition",
    when(col("Rainfall") > 0, "Rainy")
    .when(col("Snowfall") > 0, "Snowy")
    .otherwise("Normal")
)

gold_weather_impact = silver_weather_df.groupBy(
    "WeatherCondition"
).agg(
    count("*").alias("NumberOfTransactions"),
    countDistinct("InvoiceNo").alias("NumberOfInvoices"),
    spark_sum("Quantity").alias("TotalQuantity"),
    round(spark_sum("TotalPrice"), 2).alias("TotalRevenue"),
    round(avg("TotalPrice"), 2).alias("AverageLineRevenue"),
    round(avg("Temperature"), 2).alias("AverageTemperature"),
    round(spark_sum("Rainfall"), 2).alias("TotalRainfall"),
    round(spark_sum("Snowfall"), 2).alias("TotalSnowfall")
).withColumn(
    "UpdatedAt",
    current_timestamp()
).orderBy(desc("TotalRevenue"))

print("===== GOLD: WEATHER IMPACT =====")
gold_weather_impact.show(20, truncate=False)

===== GOLD: WEATHER IMPACT =====
+----------------+--------------------+----------------+-------------+------------+------------------+------------------+-------------+-------------+--------------------------+
|WeatherCondition|NumberOfTransactions|NumberOfInvoices|TotalQuantity|TotalRevenue|AverageLineRevenue|AverageTemperature|TotalRainfall|TotalSnowfall|UpdatedAt                 |
+----------------+--------------------+----------------+-------------+------------+------------------+------------------+-------------+-------------+--------------------------+
|Normal          |270223              |10391           |2837297      |5391103.76  |19.95             |12.13             |0.0          |0.0          |2026-05-07 05:57:43.099725|
|Rainy           |248105              |9258            |2665670      |5115550.47  |20.62             |11.32             |667804.2     |7171.0       |2026-05-07 05:57:43.099725|
|Snowy           |6548                |311             |69150        |134904.72   

## Gold Recommendation

### Chuẩn bị dữ liệu hóa đơn - sản phẩm

In [14]:
invoice_product_df = silver_df.filter(
    col("CustomerID") != -1
).select(
    "InvoiceNo",
    "StockCode",
    "Description"
).dropDuplicates([
    "InvoiceNo",
    "StockCode"
])

print("Số dòng Invoice-Product sau khi loại CustomerID = -1:", invoice_product_df.count())

Số dòng Invoice-Product sau khi loại CustomerID = -1: 387841


In [15]:
from pyspark.sql.functions import countDistinct, col, desc

invoice_size_df = invoice_product_df.groupBy("InvoiceNo").agg(
    countDistinct("StockCode").alias("ProductCount")
)

print("Top hóa đơn có nhiều sản phẩm nhất sau khi loại CustomerID = -1:")
invoice_size_df.orderBy(desc("ProductCount")).show(30, truncate=False)

Top hóa đơn có nhiều sản phẩm nhất sau khi loại CustomerID = -1:
+---------+------------+
|InvoiceNo|ProductCount|
+---------+------------+
|576339   |541         |
|579196   |529         |
|580727   |525         |
|578270   |439         |
|573576   |434         |
|567656   |419         |
|567183   |386         |
|575607   |375         |
|571441   |363         |
|572552   |352         |
|570488   |347         |
|568346   |333         |
|569246   |280         |
|547063   |271         |
|562031   |262         |
|570672   |259         |
|554098   |249         |
|543040   |230         |
|562046   |219         |
|569897   |210         |
|574328   |208         |
|578233   |204         |
|571653   |202         |
|556484   |195         |
|579470   |193         |
|566290   |192         |
|561894   |187         |
|580956   |184         |
|572103   |181         |
|577504   |180         |
+---------+------------+
only showing top 30 rows



In [16]:
MAX_PRODUCTS_PER_INVOICE = 1000

valid_invoice_df = invoice_size_df.filter(
    (col("ProductCount") >= 2) &
    (col("ProductCount") <= MAX_PRODUCTS_PER_INVOICE)
).select("InvoiceNo")

invoice_product_filtered_df = invoice_product_df.join(
    valid_invoice_df,
    on="InvoiceNo",
    how="inner"
)

product_pairs_df = invoice_product_filtered_df.alias("a") \
    .join(
        invoice_product_filtered_df.alias("b"),
        on="InvoiceNo",
        how="inner"
    ) \
    .filter(col("a.StockCode") < col("b.StockCode")) \
    .select(
        col("InvoiceNo"),
        col("a.StockCode").alias("ProductA"),
        col("a.Description").alias("ProductADescription"),
        col("b.StockCode").alias("ProductB"),
        col("b.Description").alias("ProductBDescription")
    )

### Tạo cặp sản phẩm mua chung trong cùng hóa đơn

In [17]:
product_pairs_df = invoice_product_filtered_df.alias("a") \
    .join(
        invoice_product_filtered_df.alias("b"),
        on="InvoiceNo",
        how="inner"
    ) \
    .filter(col("a.StockCode") < col("b.StockCode")) \
    .select(
        col("InvoiceNo"),
        col("a.StockCode").alias("ProductA"),
        col("a.Description").alias("ProductADescription"),
        col("b.StockCode").alias("ProductB"),
        col("b.Description").alias("ProductBDescription")
    )

print("Số dòng product pairs:", product_pairs_df.count())

product_pairs_df.show(10, truncate=False)

Số dòng product pairs: 9119978
+---------+--------+--------------------------------+--------+-------------------------------+
|InvoiceNo|ProductA|ProductADescription             |ProductB|ProductBDescription            |
+---------+--------+--------------------------------+--------+-------------------------------+
|536366   |22632   |HAND WARMER RED POLKA DOT       |22633   |HAND WARMER UNION JACK         |
|536386   |85099B  |JUMBO BAG RED RETROSPOT         |85099C  |JUMBO  BAG BAROQUE BLACK WHITE |
|536386   |84880   |WHITE WIRE EGG HOLDER           |85099B  |JUMBO BAG RED RETROSPOT        |
|536386   |84880   |WHITE WIRE EGG HOLDER           |85099C  |JUMBO  BAG BAROQUE BLACK WHITE |
|536398   |22637   |PIGGY BANK RETROSPOT            |22835   |HOT WATER BOTTLE I AM SO POORLY|
|536398   |22637   |PIGGY BANK RETROSPOT            |22752   |SET 7 BABUSHKA NESTING BOXES   |
|536398   |22637   |PIGGY BANK RETROSPOT            |22865   |HAND WARMER OWL DESIGN         |
|536398   |22637   

In [18]:
from pyspark.sql.functions import countDistinct, desc, col

invoice_size_df = invoice_product_filtered_df.groupBy("InvoiceNo").agg(
    countDistinct("StockCode").alias("ProductCount")
)

invoice_size_df.orderBy(desc("ProductCount")).show(10, truncate=False)

+---------+------------+
|InvoiceNo|ProductCount|
+---------+------------+
|576339   |541         |
|579196   |529         |
|580727   |525         |
|578270   |439         |
|573576   |434         |
|567656   |419         |
|567183   |386         |
|575607   |375         |
|571441   |363         |
|572552   |352         |
+---------+------------+
only showing top 10 rows



### Tính số lần mua chung

In [19]:
co_purchase_df = product_pairs_df.groupBy(
    "ProductA",
    "ProductADescription",
    "ProductB",
    "ProductBDescription"
).agg(
    countDistinct("InvoiceNo").alias("CoPurchaseCount")
)

co_purchase_df = co_purchase_df.orderBy(desc("CoPurchaseCount"))

print("Top cặp sản phẩm thường được mua chung:")
co_purchase_df.show(10, truncate=False)

Top cặp sản phẩm thường được mua chung:
+--------+---------------------------------+--------+--------------------------------+---------------+
|ProductA|ProductADescription              |ProductB|ProductBDescription             |CoPurchaseCount|
+--------+---------------------------------+--------+--------------------------------+---------------+
|22386   |JUMBO BAG PINK POLKADOT          |85099B  |JUMBO BAG RED RETROSPOT         |546            |
|22697   |GREEN REGENCY TEACUP AND SAUCER  |22699   |ROSES REGENCY TEACUP AND SAUCER |541            |
|22726   |ALARM CLOCK BAKELIKE GREEN       |22727   |ALARM CLOCK BAKELIKE RED        |530            |
|20725   |LUNCH BAG RED RETROSPOT          |22384   |LUNCH BAG PINK POLKADOT         |523            |
|20725   |LUNCH BAG RED RETROSPOT          |20727   |LUNCH BAG  BLACK SKULL.         |517            |
|82482   |WOODEN PICTURE FRAME WHITE FINISH|82494L  |WOODEN FRAME ANTIQUE WHITE      |468            |
|20725   |LUNCH BAG RED RETROSPOT

### Tính số hóa đơn của từng sản phẩm

In [20]:
product_invoice_count_df = invoice_product_df.groupBy(
    "StockCode"
).agg(
    countDistinct("InvoiceNo").alias("ProductInvoiceCount")
)

product_invoice_count_df.show(10, truncate=False)

+---------+-------------------+
|StockCode|ProductInvoiceCount|
+---------+-------------------+
|22596    |234                |
|21259    |237                |
|22728    |613                |
|21889    |449                |
|21452    |133                |
|22121    |114                |
|23318    |329                |
|21248    |52                 |
|21894    |71                 |
|90210B   |6                  |
+---------+-------------------+
only showing top 10 rows



### Tạo recommendation hai chiều

In [21]:
# Tạo chiều A → B
recommend_a_to_b = co_purchase_df.select(
    col("ProductA").alias("BaseStockCode"),
    col("ProductADescription").alias("BaseDescription"),
    col("ProductB").alias("RecommendedStockCode"),
    col("ProductBDescription").alias("RecommendedDescription"),
    col("CoPurchaseCount")
)

# Tạo chiều B → A
recommend_b_to_a = co_purchase_df.select(
    col("ProductB").alias("BaseStockCode"),
    col("ProductBDescription").alias("BaseDescription"),
    col("ProductA").alias("RecommendedStockCode"),
    col("ProductADescription").alias("RecommendedDescription"),
    col("CoPurchaseCount")
)

# Gộp hai chiều
recommendations_df = recommend_a_to_b.unionByName(recommend_b_to_a)

print("Số dòng recommendation hai chiều:", recommendations_df.count())

recommendations_df.show(10, truncate=False)

Số dòng recommendation hai chiều: 4564000
+-------------+---------------------------------+--------------------+--------------------------------+---------------+
|BaseStockCode|BaseDescription                  |RecommendedStockCode|RecommendedDescription          |CoPurchaseCount|
+-------------+---------------------------------+--------------------+--------------------------------+---------------+
|22386        |JUMBO BAG PINK POLKADOT          |85099B              |JUMBO BAG RED RETROSPOT         |546            |
|22697        |GREEN REGENCY TEACUP AND SAUCER  |22699               |ROSES REGENCY TEACUP AND SAUCER |541            |
|22726        |ALARM CLOCK BAKELIKE GREEN       |22727               |ALARM CLOCK BAKELIKE RED        |530            |
|20725        |LUNCH BAG RED RETROSPOT          |22384               |LUNCH BAG PINK POLKADOT         |523            |
|20725        |LUNCH BAG RED RETROSPOT          |20727               |LUNCH BAG  BLACK SKULL.         |517            

### Tính Confidence

In [22]:
recommendations_df = recommendations_df.join(
    product_invoice_count_df,
    recommendations_df.BaseStockCode == product_invoice_count_df.StockCode,
    how="left"
).drop("StockCode")

recommendations_df = recommendations_df.withColumn(
    "Confidence",
    round(col("CoPurchaseCount") / col("ProductInvoiceCount"), 4)
)

recommendations_df.show(10, truncate=False)

+-------------+---------------------------------+--------------------+--------------------------------+---------------+-------------------+----------+
|BaseStockCode|BaseDescription                  |RecommendedStockCode|RecommendedDescription          |CoPurchaseCount|ProductInvoiceCount|Confidence|
+-------------+---------------------------------+--------------------+--------------------------------+---------------+-------------------+----------+
|22386        |JUMBO BAG PINK POLKADOT          |85099B              |JUMBO BAG RED RETROSPOT         |546            |871                |0.6269    |
|22697        |GREEN REGENCY TEACUP AND SAUCER  |22699               |ROSES REGENCY TEACUP AND SAUCER |541            |691                |0.7829    |
|22726        |ALARM CLOCK BAKELIKE GREEN       |22727               |ALARM CLOCK BAKELIKE RED        |530            |789                |0.6717    |
|20725        |LUNCH BAG RED RETROSPOT          |22384               |LUNCH BAG PINK POLKADOT 

### Lọc recommendation yếu

In [23]:
MIN_CO_PURCHASE = 3

recommendations_df = recommendations_df.filter(
    col("CoPurchaseCount") >= MIN_CO_PURCHASE
)

print("Số dòng recommendation sau khi lọc:", recommendations_df.count())

Số dòng recommendation sau khi lọc: 1842818


### Xếp hạng top N gợi ý cho mỗi sản phẩm

In [24]:
TOP_N = 5

window_spec = Window.partitionBy("BaseStockCode") \
    .orderBy(desc("Confidence"), desc("CoPurchaseCount"))

gold_product_recommendations = recommendations_df.withColumn(
    "Rank",
    row_number().over(window_spec)
).filter(
    col("Rank") <= TOP_N
).select(
    "BaseStockCode",
    "BaseDescription",
    "RecommendedStockCode",
    "RecommendedDescription",
    "CoPurchaseCount",
    col("ProductInvoiceCount").alias("BaseProductInvoiceCount"),
    "Confidence",
    "Rank"
).orderBy(
    "BaseStockCode",
    "Rank"
)

print("Gold Product Recommendations:")
gold_product_recommendations.show(20, truncate=False)

Gold Product Recommendations:
+-------------+--------------------------+--------------------+-----------------------------------+---------------+-----------------------+----------+----+
|BaseStockCode|BaseDescription           |RecommendedStockCode|RecommendedDescription             |CoPurchaseCount|BaseProductInvoiceCount|Confidence|Rank|
+-------------+--------------------------+--------------------+-----------------------------------+---------------+-----------------------+----------+----+
|10002        |INFLATABLE POLITICAL GLOBE|20725               |LUNCH BAG RED RETROSPOT            |11             |49                     |0.2245    |1   |
|10002        |INFLATABLE POLITICAL GLOBE|22383               |LUNCH BAG SUKI  DESIGN             |9              |49                     |0.1837    |2   |
|10002        |INFLATABLE POLITICAL GLOBE|22558               |CLOTHES PEGS RETROSPOT PACK 24     |8              |49                     |0.1633    |3   |
|10002        |INFLATABLE POLITICA

## Ghi Gold tables vào HDFS

In [25]:
def write_gold_to_hdfs(df, path, table_name):
    try:
        df.write \
            .mode("overwrite") \
            .parquet(path)

        print(f"[HDFS] Đã ghi {table_name} vào {path}")

    except Exception as e:
        print(f"[HDFS] Lỗi ghi {table_name}: {e}")
        raise e

write_gold_to_hdfs(
    gold_product_recommendations,
    GOLD_PRODUCT_RECOMMENDATIONS_PATH,
    "gold_product_recommendations"
)

write_gold_to_hdfs(
    gold_sales_by_date,
    GOLD_SALES_BY_DATE_PATH,
    "gold_sales_by_date"
)

write_gold_to_hdfs(
    gold_sales_by_month,
    GOLD_SALES_BY_MONTH_PATH,
    "gold_sales_by_month"
)

write_gold_to_hdfs(
    gold_sales_by_hour,
    GOLD_SALES_BY_HOUR_PATH,
    "gold_sales_by_hour"
)

write_gold_to_hdfs(
    gold_top_products,
    GOLD_TOP_PRODUCTS_PATH,
    "gold_top_products"
)

write_gold_to_hdfs(
    gold_top_customers,
    GOLD_TOP_CUSTOMERS_PATH,
    "gold_top_customers"
)

write_gold_to_hdfs(
    gold_sales_by_country,
    GOLD_SALES_BY_COUNTRY_PATH,
    "gold_sales_by_country"
)

write_gold_to_hdfs(
    gold_holiday_impact,
    GOLD_HOLIDAY_IMPACT_PATH,
    "gold_holiday_impact"
)

write_gold_to_hdfs(
    gold_weather_impact,
    GOLD_WEATHER_IMPACT_PATH,
    "gold_weather_impact"
)

print("Đã ghi xong toàn bộ Gold Layer xuống HDFS.")

[HDFS] Đã ghi gold_product_recommendations vào hdfs://namenode:9000/data/gold/retail/product_recommendations
[HDFS] Đã ghi gold_sales_by_date vào hdfs://namenode:9000/data/gold/retail/sales_by_date
[HDFS] Đã ghi gold_sales_by_month vào hdfs://namenode:9000/data/gold/retail/sales_by_month
[HDFS] Đã ghi gold_sales_by_hour vào hdfs://namenode:9000/data/gold/retail/sales_by_hour
[HDFS] Đã ghi gold_top_products vào hdfs://namenode:9000/data/gold/retail/top_products
[HDFS] Đã ghi gold_top_customers vào hdfs://namenode:9000/data/gold/retail/top_customers
[HDFS] Đã ghi gold_sales_by_country vào hdfs://namenode:9000/data/gold/retail/sales_by_country
[HDFS] Đã ghi gold_holiday_impact vào hdfs://namenode:9000/data/gold/retail/holiday_impact
[HDFS] Đã ghi gold_weather_impact vào hdfs://namenode:9000/data/gold/retail/weather_impact
Đã ghi xong toàn bộ Gold Layer xuống HDFS.


## Ghi Gold tables vào MySQL

In [26]:
def write_gold_to_mysql(df, table_name):
    try:
        df.write.jdbc(
            url=db_url,
            table=table_name,
            mode="overwrite",
            properties=db_props
        )

        print(f"[MySQL] Đã ghi bảng {table_name}")

    except Exception as e:
        print(f"[MySQL] Lỗi ghi bảng {table_name}: {e}")
        raise e


write_gold_to_mysql(
    gold_product_recommendations,
    "gold_product_recommendations"
)

write_gold_to_mysql(
    gold_sales_by_date,
    "gold_sales_by_date"
)

write_gold_to_mysql(
    gold_sales_by_month,
    "gold_sales_by_month"
)

write_gold_to_mysql(
    gold_sales_by_hour,
    "gold_sales_by_hour"
)

write_gold_to_mysql(
    gold_top_products,
    "gold_top_products"
)

write_gold_to_mysql(
    gold_top_customers,
    "gold_top_customers"
)

write_gold_to_mysql(
    gold_sales_by_country,
    "gold_sales_by_country"
)

write_gold_to_mysql(
    gold_holiday_impact,
    "gold_holiday_impact"
)

write_gold_to_mysql(
    gold_weather_impact,
    "gold_weather_impact"
)

print("Đã ghi xong toàn bộ Gold Tables vào MySQL.")

[MySQL] Đã ghi bảng gold_product_recommendations
[MySQL] Đã ghi bảng gold_sales_by_date
[MySQL] Đã ghi bảng gold_sales_by_month
[MySQL] Đã ghi bảng gold_sales_by_hour
[MySQL] Đã ghi bảng gold_top_products
[MySQL] Đã ghi bảng gold_top_customers
[MySQL] Đã ghi bảng gold_sales_by_country
[MySQL] Đã ghi bảng gold_holiday_impact
[MySQL] Đã ghi bảng gold_weather_impact
Đã ghi xong toàn bộ Gold Tables vào MySQL.


## =========TEST=========

In [27]:
check_recommendations = spark.read.parquet(GOLD_PRODUCT_RECOMMENDATIONS_PATH)

print("===== CHECK: GOLD PRODUCT RECOMMENDATIONS FROM HDFS =====")
check_recommendations.show(20, truncate=True)

===== CHECK: GOLD PRODUCT RECOMMENDATIONS FROM HDFS =====
+-------------+--------------------+--------------------+----------------------+---------------+-----------------------+----------+----+
|BaseStockCode|     BaseDescription|RecommendedStockCode|RecommendedDescription|CoPurchaseCount|BaseProductInvoiceCount|Confidence|Rank|
+-------------+--------------------+--------------------+----------------------+---------------+-----------------------+----------+----+
|        10002|INFLATABLE POLITI...|               20725|  LUNCH BAG RED RET...|             11|                     49|    0.2245|   1|
|        10002|INFLATABLE POLITI...|               22383|  LUNCH BAG SUKI  D...|              9|                     49|    0.1837|   2|
|        10002|INFLATABLE POLITI...|               22558|  CLOTHES PEGS RETR...|              8|                     49|    0.1633|   3|
|        10002|INFLATABLE POLITI...|               22367|  CHILDRENS APRON S...|              8|                     49|

In [28]:
check_recommendations = spark.read.parquet(GOLD_PRODUCT_RECOMMENDATIONS_PATH)

print("===== CHECK: GOLD PRODUCT RECOMMENDATIONS FROM HDFS =====")
check_recommendations.show(20, truncate=True)

===== CHECK: GOLD PRODUCT RECOMMENDATIONS FROM HDFS =====
+-------------+--------------------+--------------------+----------------------+---------------+-----------------------+----------+----+
|BaseStockCode|     BaseDescription|RecommendedStockCode|RecommendedDescription|CoPurchaseCount|BaseProductInvoiceCount|Confidence|Rank|
+-------------+--------------------+--------------------+----------------------+---------------+-----------------------+----------+----+
|        10002|INFLATABLE POLITI...|               20725|  LUNCH BAG RED RET...|             11|                     49|    0.2245|   1|
|        10002|INFLATABLE POLITI...|               22383|  LUNCH BAG SUKI  D...|              9|                     49|    0.1837|   2|
|        10002|INFLATABLE POLITI...|               22558|  CLOTHES PEGS RETR...|              8|                     49|    0.1633|   3|
|        10002|INFLATABLE POLITI...|               22367|  CHILDRENS APRON S...|              8|                     49|